# MLP Analisys

In [1]:
# === STEP 1: Import e configurazione Spark ===

# Standard library
import time
import json
from itertools import product

# Spark core
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StringType, StructType, StructField

# Spark MLlib — feature engineering
from pyspark.ml.feature import VectorAssembler, StandardScaler

# Spark MLlib — modello
from pyspark.ml.classification import MultilayerPerceptronClassifier, MultilayerPerceptronClassificationModel
# Spark MLlib — valutazione
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

# Spark MLlib — pipeline
from pyspark.ml import Pipeline
from pyspark.ml import PipelineModel

# Pucktrick
from pucktrick import PuckTrick, Engine

# Solo per export finale — NON usato nel loop sperimentale
import pandas as pd

## Step 2 — Configurazione Spark e caricamento dataset

In questa sezione inizializziamo la `SparkSession` connettendoci al cluster remoto e carichiamo il dataset **MetroPT-3**.

Il preprocessing si limita a:
- **Selezione** delle sole colonne necessarie agli esperimenti: le 3 feature ad alta importanza identificate nell'analisi esplorativa (`DV_pressure`, `Oil_temperature`, `TP3`) e la label `target`
- **Cast** di tutte le colonne a `DoubleType`, requisito di Spark MLlib per feature e label
- **Drop dei null** per garantire consistenza tra i run

Infine, inizializziamo `PuckTrick` con backend Spark: questo aggiunge automaticamente la colonna `_pucktrick_row_id` al DataFrame, necessaria per tracciare le righe modificate durante l'iniezione del rumore. Il DataFrame risultante (`base_df`) è la **base immutabile** da cui partono tutti gli esperimenti — non verrà mai modificato direttamente.

### Costanti sperimentali

| Parametro | Valore |
|---|---|
| Feature selezionate | `DV_pressure_scaled`, `Oil_temperature_scaled`, `TP3_scaled` |
| Tipi di rumore | `duplicated`, `labels`, `missing`, `noisy`, `outliers` |
| Livelli di rumore | 0%, 10%, 20%, 30%, 50% |
| Run per combinazione | 20 |
| t di Student (95%, df=19) | 2.093 |

### Nota statistica — Intervallo di Confidenza al 95%

Per ogni combinazione *(tipo di rumore × feature × percentuale)* eseguiamo **20 run indipendenti**,
ciascuno con un seed diverso. Questo ci permette di stimare non solo la metrica media,
ma anche la sua **variabilità** tramite l'intervallo di confidenza al 95%.

La formula utilizzata è quella dell'intervallo di confidenza per campioni piccoli (distribuzione t di Student):

$$IC = \bar{x} \pm t \cdot \frac{s}{\sqrt{n}}$$

Dove:
- $\bar{x}$ è la **media** dell'F1-score sui 20 run
- $s$ è la **deviazione standard** campionaria
- $n = 20$ è il numero di run
- $t = 2.093$ è il valore critico dalla **tavola t di Student** con $df = n - 1 = 19$ gradi di libertà e livello di confidenza al 95%

**Esempio pratico:** se sui 20 run otteniamo F1 medio $= 0.85$ con deviazione standard $= 0.03$:

$$IC = 0.85 \pm 2.093 \cdot \frac{0.03}{\sqrt{20}} = 0.85 \pm 0.014 = [0.836,\ 0.864]$$

Questo significa che siamo al 95% confidenti che il vero valore dell'F1 — quello che otterremmo
con infiniti run — cada nell'intervallo $[0.836, 0.864]$.

Confrontando gli intervalli tra livelli di rumore diversi, possiamo stabilire se una degradazione
delle performance è **statisticamente significativa** o rientra nella variabilità naturale del modello.

> **Nota:** il valore $t = 2.093$ è valido **solo con $n = 20$ run**. Se si modifica `N_RUNS`,
> aggiornare `T_VALUE_95` consultando la tavola t di Student alla riga $df = N\_RUNS - 1$.

Link alla tavola student
https://www.statisticshowto.com/tables/t-distribution-table/

In [2]:
# === STEP 2: Configurazione Spark e caricamento dataset ===

# ── Configurazione cluster ─────────────────────────────────────────────────
MASTER_URL  = "spark://10.0.1.8:7077"
DRIVER_HOST = "10.0.1.8"

spark = SparkSession.builder \
    .appName("MetroPT_MLP_Experiments") \
    .master(MASTER_URL) \
    .config("spark.submit.deployMode",      "client") \
    .config("spark.executor.instances",     "4") \
    .config("spark.executor.cores",         "4") \
    .config("spark.executor.memory",        "13g") \
    .config("spark.driver.memory",          "8g") \
    .config("spark.driver.host",            DRIVER_HOST) \
    .config("spark.driver.bindAddress",     DRIVER_HOST) \
    .config("spark.sql.shuffle.partitions", "32") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("SparkSession creata — versione:", spark.version)

# ── Costanti sperimentali ──────────────────────────────────────────────────
ALL_FEATURES = [
    "TP2_scaled", "TP3_scaled", "H1_scaled", "DV_pressure_scaled",
    "Reservoirs_scaled", "Oil_temperature_scaled", "Motor_current_scaled",
    "COMP", "DV_eletric", "Towers", "MPG", "LPS", "Pressure_switch"
]
NOISE_FEATURES = ["DV_pressure_scaled", "Oil_temperature_scaled", "TP3_scaled"]
LABEL_COL    = "target"
NOISE_TYPES = ["duplicated", "labels", "missing", "noisy", "outliers"]
NOISE_LEVELS = [0.0, 0.1, 0.2, 0.3, 0.5]
N_RUNS       = 20
T_VALUE_95   = 2.093

# ── Caricamento dataset ────────────────────────────────────────────────────
# Nota: il file è stato salvato con pandas to_parquet (file singolo, non directory Spark).
# Si legge sul driver e si distribuisce immediatamente sul cluster via createDataFrame.
DATA_PATH = "/home/PuckTrickadmin/DATASETS/MetroDT_Modified.parquet"
raw_pdf = pd.read_parquet(DATA_PATH)
raw_df = spark.createDataFrame(raw_pdf)
del raw_pdf  # libera subito memoria driver

# ── Selezione colonne e cast ───────────────────────────────────────────────
df = raw_df.select(
    F.col("timestamp"),
    F.col("TP2_scaled").cast(DoubleType()),
    F.col("TP3_scaled").cast(DoubleType()),
    F.col("H1_scaled").cast(DoubleType()),
    F.col("DV_pressure_scaled").cast(DoubleType()),
    F.col("Reservoirs_scaled").cast(DoubleType()),
    F.col("Oil_temperature_scaled").cast(DoubleType()),
    F.col("Motor_current_scaled").cast(DoubleType()),
    F.col("COMP").cast(DoubleType()),
    F.col("DV_eletric").cast(DoubleType()),
    F.col("Towers").cast(DoubleType()),
    F.col("MPG").cast(DoubleType()),
    F.col("LPS").cast(DoubleType()),
    F.col("Pressure_switch").cast(DoubleType()),
    F.col("target").cast(DoubleType())
).dropna()
# ── Split temporale — coerente con il preprocessing ───────────────────────
SPLIT_DATE = "2020-06-01 00:00:00"

pt         = PuckTrick(df, engine=Engine.SPARK)
base_df    = pt.original

base_train_df = base_df.filter(F.col("timestamp") < SPLIT_DATE).drop("timestamp")
base_test_df  = base_df.filter(F.col("timestamp") >= SPLIT_DATE).drop("timestamp")

base_train_df.cache()
base_test_df.cache()

n_train       = base_train_df.count()
n_test        = base_test_df.count()
n_train_fault = base_train_df.filter(F.col("target") == 1).count()
n_test_fault  = base_test_df.filter(F.col("target") == 1).count()

print(f"Training : {n_train:,} righe ({n_train_fault:,} guasti, {n_train_fault/n_train*100:.2f}%)")
print(f"Test     : {n_test:,} righe ({n_test_fault:,} guasti, {n_test_fault/n_test*100:.2f}%)")
print(f"Split    : {n_train/(n_train+n_test)*100:.1f}% train / {n_test/(n_train+n_test)*100:.1f}% test")
print(f"\nAtteso dal preprocessing:")
print(f"  Training: ~856,832 righe, ~1.29% guasti")
print(f"  Test    : ~660,116 righe, ~2.87% guasti")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/03 08:24:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession creata — versione: 4.1.1


[2026-03-03 08:26:12] [INFO] Inizializzazione PuckTrick...


[2026-03-03 08:26:12] [INFO] Backend richiesto: Engine.SPARK


[2026-03-03 08:26:12] [DEBUG] PySpark availability: True


...

[2026-03-03 08:26:12] [INFO] SparkBackend pronto.


[2026-03-03 08:26:12] [INFO] Backend attivo: Engine.SPARK



[Stage 0:=================>                                       (5 + 11) / 16]




[Stage 4:==============>                                          (4 + 12) / 16]



Training : 856,832 righe (11,017 guasti, 1.29%)
Test     : 660,116 righe (18,937 guasti, 2.87%)
Split    : 56.5% train / 43.5% test

Atteso dal preprocessing:
  Training: ~856,832 righe, ~1.29% guasti
  Test    : ~660,116 righe, ~2.87% guasti


## Step 3 — Tuning iperparametri MLP sul dato pulito

### Perché il tuning è necessario

Il `MultilayerPerceptronClassifier` di Spark MLlib espone diversi iperparametri
che influenzano significativamente le performance del modello. Sceglierli in modo
arbitrario rischierebbe di introdurre un bias nei risultati: un modello mal
configurato potrebbe degradare non per il rumore introdotto, ma semplicemente
perché l'architettura è inadatta al problema.

Per questo motivo eseguiamo una sessione di tuning **una volta sola, sul dato
pulito**, e fissiamo gli iperparametri trovati per tutti i 3.000 run sperimentali.
Questo garantisce che qualsiasi variazione di F1-score osservata nei capitoli
successivi sia attribuibile **esclusivamente al rumore** e non a differenze di
configurazione del modello.

### Architettura del MLP

Il `MultilayerPerceptronClassifier` di Spark MLlib implementa un percettrone
multistrato con funzione di attivazione **sigmoide** per gli strati nascosti e
**softmax** per lo strato di output. Il parametro `layers` definisce la struttura
completa della rete come lista di interi, dove ogni elemento rappresenta il numero
di neuroni in quello strato.

Nel nostro caso:
- **Input layer**: dimensione fissa **3**, una per ciascuna feature
  (`DV_pressure_scaled`, `Oil_temperature_scaled`, `TP3_scaled`)
- **Hidden layer(s)**: da ottimizzare
- **Output layer**: dimensione fissa **2**, una per ciascuna classe
  (classe 0 = normale, classe 1 = guasto)

### Iperparametri da ottimizzare

| Iperparametro | Descrizione | Valori testati |
|---|---|---|
| `layers` | Architettura completa della rete | `[13,16,2]`, `[13,32,2]`, `[13,64,2]`, `[13,32,16,2]`, `[13,64,32,16,2]` |
| `maxIter` | Numero massimo di iterazioni di ottimizzazione | `50`, `100` |
| `stepSize` | Learning rate dell'ottimizzatore L-BFGS | `0.01`, `0.05` |
| `blockSize` | Dimensione del blocco per aggregazione gradienti | `128` (fisso) |

> **Nota su `blockSize`:** controlla quanti campioni vengono aggregati insieme
> prima di aggiornare i pesi. Valori più alti migliorano l'efficienza su cluster
> ma richiedono più memoria per executor. `128` è il default consigliato da Spark
> per dataset di questa dimensione.

> **Nota su `stepSize`:** a differenza del learning rate nel SGD classico, qui
> controlla il passo iniziale dell'ottimizzatore **L-BFGS** usato internamente
> da Spark MLlib. L-BFGS è un metodo quasi-Newton che converge più velocemente
> del gradient descent per problemi con poche feature — adatto al nostro caso
> con sole 3 feature in input.

### Strategia di ricerca

La ricerca avviene tramite **`CrossValidator`** con **3 fold** applicato su un
campione del **20% del training set**. Ridurre il dataset per il tuning è una
scelta deliberata: con 856.832 righe, eseguire 16 combinazioni × 3 fold sul
dataset completo richiederebbe tempi eccessivi senza un reale beneficio,
dato che il modello converge su campioni molto più piccoli.

Il test set è **completamente escluso** dal tuning — viene usato solo alla fine
di questo step per calcolare la **baseline F1** sul dato pulito a 0% di rumore,
che costituisce il riferimento per tutti i risultati del Capitolo 4.

La metrica di selezione è **F1-score weighted**: tiene conto dello sbilanciamento
tra classi (98% normale, 2% guasto) pesando il contributo di ciascuna classe
proporzionalmente alla sua frequenza, senza ignorare la classe minoritaria come
farebbe l'accuracy.

### Output atteso

Al termine di questo step otteniamo:
- Le costanti `BEST_LAYERS`, `BEST_MAX_ITER`, `BEST_STEP` — fissate per tutti i run successivi
- La **baseline F1** sul test set pulito — il punto di riferimento del Capitolo 4

In [ ]:
# === STEP 3: Tuning iperparametri MLP sul dato pulito ===
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml import PipelineModel
from pyspark.ml.classification import MultilayerPerceptronClassificationModel
import json

# ── Campione per il tuning (20% del training set) ─────────────────────────
tune_df, _ = base_train_df.randomSplit([0.2, 0.8], seed=42)
tune_df.cache()
print(f"Righe per tuning: {tune_df.count():,}")

# ── VectorAssembler ────────────────────────────────────────────────────────
assembler = VectorAssembler(
    inputCols    = ALL_FEATURES,
    outputCol    = "features",
    handleInvalid= "keep"
)

# ── Modello base ───────────────────────────────────────────────────────────
mlp = MultilayerPerceptronClassifier(
    featuresCol = "features",
    labelCol    = LABEL_COL,
    seed        = 42
)

# ── Pipeline ───────────────────────────────────────────────────────────────
pipeline = Pipeline(stages=[assembler, mlp])

# ── Griglia iperparametri ──────────────────────────────────────────────────
param_grid = ParamGridBuilder() \
    .addGrid(mlp.layers, [
        [13, 64, 2],
        [13, 32, 16, 2],
        [13, 64, 32, 16, 2]
    ]) \
    .addGrid(mlp.maxIter,    [50, 100,200,300,400]) \
    .addGrid(mlp.stepSize,   [0.01, 0.05]) \
    .addGrid(mlp.blockSize,  [128, 256, 512]) \
    .build()

print(f"Combinazioni da testare: {len(param_grid)}")

# ── Evaluator ──────────────────────────────────────────────────────────────
evaluator = MulticlassClassificationEvaluator(
    labelCol      = LABEL_COL,
    predictionCol = "prediction",
    metricName    = "f1"
)

# ── CrossValidator ─────────────────────────────────────────────────────────
cv = CrossValidator(
    estimator        = pipeline,
    estimatorParamMaps = param_grid,
    evaluator        = evaluator,
    numFolds         = 3,
    seed             = 42,
    parallelism      = 4
)

# ── Esecuzione tuning ──────────────────────────────────────────────────────
print("Avvio tuning...")
t0 = time.time()
cv_model = cv.fit(tune_df)
print(f"Tuning completato in {(time.time()-t0)/60:.1f} minuti")

# ── Estrazione migliori iperparametri ──────────────────────────────────────
best_pipeline_model: PipelineModel = cv_model.bestModel  # type: ignore
best_mlp: MultilayerPerceptronClassificationModel = best_pipeline_model.stages[-1]  # type: ignore

BEST_LAYERS    = best_mlp.getLayers()
BEST_MAX_ITER  = best_mlp.getMaxIter()
BEST_STEP      = best_mlp.getStepSize()
BEST_BLOCK     = best_mlp.getBlockSize()

print("\n=== MIGLIORI IPERPARAMETRI ===")
print(f"  layers   : {BEST_LAYERS}")
print(f"  maxIter  : {BEST_MAX_ITER}")
print(f"  stepSize : {BEST_STEP}")
print(f"  blockSize: {BEST_BLOCK}")

# ── Baseline F1 sul test set pulito ───────────────────────────────────────
baseline_pred = best_pipeline_model.transform(base_test_df)
baseline_f1   = evaluator.evaluate(baseline_pred)

print(f"\n=== BASELINE (0% rumore) ===")
print(f"  F1-score sul test set pulito: {baseline_f1:.4f}")

# ── Salvataggio ────────────────────────────────────────────────────────────
PARAMS_PATH = "/home/PuckTrickadmin/Daniel/PARAMS/mlp_best_params.json"

model_params = {
    "layers":      list(BEST_LAYERS),
    "maxIter":     BEST_MAX_ITER,
    "stepSize":    BEST_STEP,
    "blockSize":   BEST_BLOCK,
    "baseline_f1": baseline_f1
}

with open(PARAMS_PATH, "w") as f:
    json.dump(model_params, f, indent=2)

print(f"\n✅ Parametri salvati in: {PARAMS_PATH}")
print(json.dumps(model_params, indent=2))

In [4]:
#derivate dall'output generato
BEST_LAYERS = [13,64,2]
BEST_MAX_ITER = 100
BEST_STEP = 0.05
BEST_BLOCK = 128
BASELINE_F1 = 0.9822523417285991

---

## Step 4 — Funzione `inject_noise()`

### Obiettivo

In questo step definiamo la funzione `inject_noise()` — il wrapper attorno a
**Pucktrick** che verrà chiamato ad ogni run del loop sperimentale principale.

La funzione ha un compito preciso: dato il training set pulito, un tipo di rumore,
una feature target e una percentuale, restituisce un nuovo DataFrame con il rumore
iniettato secondo la strategia specificata. Il test set non viene mai toccato.

### Strategia di iniezione

Per ogni tipo di rumore, Pucktrick richiede una `strategy` — un dizionario che
specifica come e dove applicare la corruzione. I parametri chiave sono:

| Parametro | Descrizione |
|---|---|
| `affected_features` | Lista delle colonne su cui applicare il rumore |
| `selection_criteria` | Filtro sulle righe — `"all"` per applicare su tutto il dataset |
| `percentage` | Percentuale di righe da corrompere (es. `0.1` per 10%) |
| `mode` | Modalità di iniezione — vedi sotto |
| `perturbate_data` | Configurazione della distribuzione di campionamento |

### Modalità `mode="new"` vs `mode="extended"`

Pucktrick supporta due modalità di iniezione:

- **`mode="new"`** — applica il rumore indipendentemente dallo stato attuale del
  DataFrame. Ogni chiamata è indipendente e ignora eventuali modifiche pregresse.
  È la modalità corretta per il nostro caso perché **partiamo sempre dal
  `base_train_df` pulito** ad ogni run — non c'è rumore pregresso da considerare.

- **`mode="extended"`** — tiene traccia delle righe già modificate e aggiunge
  solo il rumore necessario per raggiungere la percentuale target in modo
  cumulativo. Utile quando si vuole incrementare progressivamente il rumore
  su un DataFrame già parzialmente corrotto.

### I 5 tipi di rumore e il loro comportamento

**`duplicated`** — Duplica righe esistenti aggiungendole al DataFrame. Distorce
la distribuzione del dataset aumentando artificialmente la frequenza di certi
campioni. Sul training set questo può causare overfitting su pattern specifici.

**`labels`** — Inverte le etichette di classificazione su una percentuale di righe
(da 0 a 1 o viceversa). È il tipo di rumore più dannoso teoricamente: il modello
impara pattern sbagliati associando feature di classe normale a guasto e viceversa.

**`missing`** — Introduce valori `NaN` nelle colonne selezionate. I valori nulli
vengono gestiti direttamente dal `VectorAssembler` tramite il parametro
`handleInvalid="keep"` che sostituisce i null con `0.0` nel vettore delle feature,
senza richiedere un passaggio di imputazione separato. Questa scelta va documentata
nel Capitolo 3: la sostituzione con `0.0` introduce un bias verso lo zero che
è parte integrante dell'effetto del rumore `missing` sul modello.

**`noisy`** — Aggiunge rumore numerico ai valori delle feature sostituendoli con
valori casuali nell'intervallo `[min, max]` della colonna. Degrada il segnale
progressivamente — il modello riceve feature sempre meno informative.

**`outliers`** — Introduce valori anomali oltre 3 deviazioni standard dalla media.
A differenza del rumore gaussiano, gli outlier sono perturbazioni estreme e
localizzate — possono influenzare fortemente le statistiche usate per la
normalizzazione.

### Gestione dei valori nulli

`VectorAssembler` di Spark MLlib espone il parametro `handleInvalid` per gestire
i valori nulli direttamente nella pipeline:

- `"error"` — default, lancia eccezione in presenza di null
- `"skip"` — elimina le righe con null dal DataFrame
- `"keep"` — sostituisce i null con `0.0` nel vettore

Usiamo `"keep"` per tutti i run — anche quelli senza rumore `missing` — così
la pipeline è uniforme e non richiede rami condizionali.

### Seed per riproducibilità

Ad ogni run del loop sperimentale viene passato un `seed` diverso alla funzione.
Il seed controlla il campionamento di Pucktrick — quali righe specifiche vengono
corrotte. Variare il seed ad ogni run garantisce che i 20 run siano statisticamente
indipendenti e che l'intervallo di confidenza al 95% sia valido.

### Firma della funzione
```python
def inject_noise(
    pt,              # istanza PuckTrick già inizializzata
    train_df,        # DataFrame di training pulito (base_train_df senza timestamp)
    noise_type,      # str: "duplicated"|"labels"|"missing"|"noisy"|"outliers"
    feature,         # str: feature su cui iniettare il rumore (da NOISE_FEATURES)
    percentage,      # float: percentuale di righe da corrompere (0.0–0.5)
    seed             # int: seed per riproducibilità del campionamento
) -> DataFrame:      # DataFrame con rumore iniettato, pronto per il training
```

> **Nota:** quando `percentage=0.0` la funzione restituisce il DataFrame originale
> senza modifiche — questo gestisce automaticamente la baseline (0% rumore) nel
> loop sperimentale senza richiedere un ramo separato.

In [ ]:
# # === STEP 4: Funzione inject_noise() ===

# def inject_noise(pt, train_df, noise_type, feature, percentage, seed):
#     """
#     Inietta rumore controllato su una feature del training set tramite Pucktrick.

#     Args:
#         pt          : istanza PuckTrick già inizializzata su base_train_df
#         train_df    : DataFrame di training pulito (deve contenere ROW_ID)
#         noise_type  : tipo di rumore da iniettare
#         feature     : feature su cui applicare il rumore
#         percentage  : percentuale di righe da corrompere (0.0 = nessun rumore)
#         seed        : seed per riproducibilità del campionamento

#     Returns:
#         DataFrame con rumore iniettato, pronto per il training
#     """

#     # Baseline — nessun rumore
#     if percentage == 0.0:
#         return train_df

#     # Strategia comune a tutti i tipi di rumore
#     strategy = {
#         "affected_features": [feature],
#         "selection_criteria": "all",
#         "percentage":         percentage,
#         "mode":               "new",
#         "perturbate_data": {
#             "distribution": "random",
#             "param":        {"seed": seed}
#         }
#     }

#     # Dispatch per tipo di rumore
#     if noise_type == "duplicated":
#         status, noisy_df = pt.duplicated(train_df, strategy)
#     elif noise_type == "labels":
#         label_strategy = {**strategy, "affected_features": [LABEL_COL]}
#         status, noisy_df = pt.labels(train_df, label_strategy)
#     elif noise_type == "missing":
#         status, noisy_df = pt.missing(train_df, strategy)
#     elif noise_type == "noisy":
#         status, noisy_df = pt.noise(train_df, strategy)
#     elif noise_type == "outliers":
#         status, noisy_df = pt.outlier(train_df, strategy)
#     else:
#         raise ValueError(f"Tipo di rumore non supportato: {noise_type}")

#     if status != 0:
#         print(f"⚠️  inject_noise: status={status} "
#                 f"({noise_type}, {feature}, {percentage:.0%}, seed={seed})")
#         return train_df

#     return noisy_df


# # ── Inizializzazione PuckTrick sul training set ────────────────────────────
# pt_train = PuckTrick(base_train_df, engine=Engine.SPARK)

# # # ── Test rapido ────────────────────────────────────────────────────────────
# # print("Test inject_noise()...")

# # for noise_type in NOISE_TYPES:
# #     test_noisy = inject_noise(
# #         pt         = pt_train,
# #         train_df   = pt_train.original,
# #         noise_type = noise_type,
# #         feature    = "DV_pressure_scaled",
# #         percentage = 0.1,
# #         seed       = 42
# #     )
# #     n_orig  = pt_train.original.count()
# #     n_noisy = test_noisy.count()
# #     print(f"  ✅ {noise_type:<12} | righe: {n_orig:,} → {n_noisy:,}")

# # print("\nTest completato.")

[2026-03-03 08:26:20] [INFO] Inizializzazione PuckTrick...


[2026-03-03 08:26:20] [INFO] Backend richiesto: Engine.SPARK


[2026-03-03 08:26:20] [DEBUG] PySpark availability: True


[2026-03-03 08:26:20] [INFO] Forzo backend Spark.


[2026-03-03 08:26:20] [INFO] Creazione SparkBackend...


...

[2026-03-03 08:26:20] [DEBUG] SparkSession già esistente.


[2026-03-03 08:26:20] [DEBUG] SparkSession già esistente.


[2026-03-03 08:26:20] [INFO] SparkBackend pronto.


[2026-03-03 08:26:20] [INFO] SparkBackend pronto.


[2026-03-03 08:26:20] [INFO] Backend attivo: Engine.SPARK


## Step 5 — Funzione `run_experiment()`

### Obiettivo

In questo step definiamo la funzione `run_experiment()` — il cuore del loop
sperimentale. Questa funzione esegue **un singolo run** dell'esperimento:
dato un tipo di rumore, una feature, una percentuale e un seed, restituisce
le metriche di performance del modello addestrato su dati rumorosi e valutato
sul test set pulito.

### Flusso di un singolo run
```
base_train_df (pulito)
        ↓
inject_noise(noise_type, feature, percentage, seed)
        ↓
noisy_train_df
        ↓
VectorAssembler → features vector
        ↓
MultilayerPerceptronClassifier.fit()
        ↓
model.transform(base_test_df)
        ↓
F1-score + AUC-ROC
```

Il test set è **sempre pulito e fisso** — le metriche misurano esclusivamente
la capacità del modello addestrato su dati rumorosi di generalizzare su dati
reali non corrotti. Questo è il design corretto per simulare uno scenario
realistico: i dati storici di training sono rumorosi, i dati nuovi in produzione
no.

### Metriche calcolate

Per ogni run calcoliamo due metriche complementari:

**F1-score weighted** — metrica principale. Tiene conto dello sbilanciamento
tra classi (98% normale, 2% guasto) pesando il contributo di ciascuna classe
proporzionalmente alla sua frequenza. È la metrica usata per il tuning e per
il confronto tra run.

**AUC-ROC** — metrica di controllo. Misura la capacità discriminativa del
modello indipendentemente dalla soglia di classificazione. È particolarmente
informativa su dataset sbilanciati perché non è influenzata dalla distribuzione
delle classi.

### Pipeline di training

Ad ogni run costruiamo una pipeline Spark MLlib con:

1. **`VectorAssembler`** — assembla le 13 feature in un unico vettore con
   `handleInvalid="keep"` per gestire i NaN introdotti dal rumore `missing`
   sostituendoli con `0.0`
2. **`MultilayerPerceptronClassifier`** — con gli iperparametri fissi trovati
   nel tuning: `layers=[13,64,2]`, `maxIter=100`, `stepSize=0.05`, `blockSize=128`

Il seed del modello viene variato ad ogni run in modo deterministico:
`model_seed = seed * 100` — questo garantisce che sia il campionamento del
rumore che l'inizializzazione dei pesi della rete siano diversi ad ogni run,
massimizzando l'indipendenza statistica tra i 20 run.

### Output della funzione

La funzione restituisce un dizionario con tutte le informazioni necessarie
per ricostruire i risultati:
```python
{
    "noise_type":  str,    # tipo di rumore
    "feature":     str,    # feature corrotta
    "percentage":  float,  # percentuale di rumore
    "seed":        int,    # seed usato
    "f1":          float,  # F1-score weighted sul test set
    "auc":         float,  # AUC-ROC sul test set
    "n_train":     int,    # righe nel training set dopo iniezione
    "duration_s":  float   # tempo di esecuzione in secondi
}
```

### Nota sul tempo di esecuzione

Con 2.500 run totali e un tempo stimato di 15-30 secondi per run su cluster,
il loop sperimentale completo richiederà circa **10-20 ore**. Per questo motivo
i risultati vengono salvati in modo incrementale su disco dopo ogni run —
se il processo viene interrotto, è possibile riprendere dall'ultimo run completato
senza perdere il lavoro fatto.

### Ottimizzazioni per il loop sperimentale

Per contenere i tempi del loop sperimentale (~2.500 run) sono state applicate
due ottimizzazioni alla funzione `run_experiment()`:

**Ottimizzazione 1 — Pipeline pre-costruita:** `VectorAssembler` e
`MultilayerPerceptronClassifier` vengono costruiti **una volta sola** fuori
dalla funzione e passati come parametri. Evita di ricreare gli oggetti pipeline
2.500 volte.

**Ottimizzazione 2 — `n_train` condizionale:** il conteggio delle righe del
training set dopo l'iniezione viene eseguito solo per `duplicated` — l'unico
tipo di rumore che modifica il numero di righe. Per tutti gli altri tipi il
conteggio è identico al training set originale e viene letto direttamente senza
chiamate aggiuntive al cluster.

In [ ]:
# # === STEP 5: Funzione run_experiment() — ottimizzata ===
# from pyspark.ml.feature import Imputer

# # ── Oggetti pre-costruiti — creati una volta sola ─────────────────────────
# imputer = Imputer(
#     inputCols  = ALL_FEATURES,
#     outputCols = ALL_FEATURES,
#     strategy   = "mean"
# )

# base_assembler = VectorAssembler(
#     inputCols    = ALL_FEATURES,
#     outputCol    = "features",
#     handleInvalid= "skip"   # skip le righe ancora problematiche come safety net
# )

# base_mlp = MultilayerPerceptronClassifier(
#     featuresCol = "features",
#     labelCol    = LABEL_COL,
#     layers      = BEST_LAYERS,
#     maxIter     = BEST_MAX_ITER,
#     stepSize    = BEST_STEP,
#     blockSize   = BEST_BLOCK
# )

# f1_evaluator = MulticlassClassificationEvaluator(
#     labelCol      = LABEL_COL,
#     predictionCol = "prediction",
#     metricName    = "f1"
# )

# auc_evaluator = BinaryClassificationEvaluator(
#     labelCol         = LABEL_COL,
#     rawPredictionCol = "rawPrediction",
#     metricName       = "areaUnderROC"
# )

# N_TRAIN_BASE = pt_train.original.count()
# print(f"✅ N_TRAIN_BASE: {N_TRAIN_BASE:,}")

# def run_experiment(pt, noise_type, feature, percentage, seed):
#     t0 = time.time()
#     noisy_train = inject_noise(
#         pt         = pt,
#         train_df   = pt.original,
#         noise_type = noise_type,
#         feature    = feature,
#         percentage = percentage,
#         seed       = seed
#     )

#     mlp_run  = base_mlp.setSeed(seed * 100)
#     pipeline = Pipeline(stages=[imputer, base_assembler, mlp_run])
#     model    = pipeline.fit(noisy_train)

#     predictions = model.transform(base_test_df)
#     f1  = f1_evaluator.evaluate(predictions)
#     auc = auc_evaluator.evaluate(predictions)
#     duration = time.time() - t0
#     n_train  = noisy_train.count() if noise_type == "duplicated" else N_TRAIN_BASE

#     return {
#         "noise_type": noise_type,
#         "feature":    feature,
#         "percentage": percentage,
#         "seed":       seed,
#         "f1":         f1,
#         "auc":        auc,
#         "n_train":    n_train,
#         "duration_s": duration
#     }

✅ N_TRAIN_BASE: 856,832


In [7]:
# # === PROFILING TEMPI ===
# print("Profiling run_experiment()...")

# t0 = time.time()
# noisy_train = inject_noise(
#     pt         = pt_train,
#     train_df   = pt_train.original,
#     noise_type = "noisy",
#     feature    = "DV_pressure_scaled",
#     percentage = 0.1,
#     seed       = 42
# )
# t1 = time.time()
# print(f"  inject_noise : {t1-t0:.1f}s")

# mlp_run  = base_mlp.setSeed(42 * 100)
# pipeline = Pipeline(stages=[base_assembler, mlp_run])
# model    = pipeline.fit(noisy_train)
# t2 = time.time()
# print(f"  pipeline.fit : {t2-t1:.1f}s")

# predictions = model.transform(base_test_df)
# f1  = f1_evaluator.evaluate(predictions)
# auc = auc_evaluator.evaluate(predictions)
# t3 = time.time()
# print(f"  evaluate     : {t3-t2:.1f}s")
# print(f"  totale       : {t3-t0:.1f}s")

## Step 6 — Loop sperimentale principale

### Struttura del loop

Il loop esegue tutte le combinazioni previste dal piano sperimentale:
```
Per ogni TIPO DI ERRORE (5):
    Se noise_type == "labels":
        Per ogni PERCENTUALE (5):
            Per ogni RUN (20):
                run_experiment()
    Altrimenti:
        Per ogni FEATURE (3):
            Per ogni PERCENTUALE (5):
                Per ogni RUN (20):
                    run_experiment()
```

Totale run stimati: `(4 × 3 × 5 × 20) + (1 × 5 × 20) = 2.400 + 100 = 2.500 run`

### Salvataggio incrementale

I risultati vengono salvati su disco dopo ogni run in formato JSON lines
(un dizionario per riga). Questo garantisce che in caso di interruzione
del processo, i run già completati non vengano persi e il loop possa
riprendere dall'ultimo checkpoint.

### Ripresa automatica

All'avvio il loop verifica se esiste già un file di risultati parziali.
Se esiste, carica i run già completati e salta le combinazioni già eseguite
— il loop è quindi **idempotente**: può essere interrotto e rilanciato
senza duplicare i risultati.

### Stima tempi

| Scenario | Tempo stimato |
|---|---|
| 60s per run × 2.500 run | ~41 ore |
| Con overhead e variabilità | ~45-48 ore |

> **Raccomandazione:** lanciare il loop la sera con `nohup` o in una sessione
> `tmux` per evitare interruzioni dovute alla disconnessione SSH.

In [ ]:
# === STEP 6: Loop sperimentale principale ===
import json
import os
from datetime import datetime

RESULTS_PATH = "/home/PuckTrickadmin/Daniel/RESULTS/mlp_results.jsonl"
os.makedirs(os.path.dirname(RESULTS_PATH), exist_ok=True)

# Tipi di rumore che non iterano sulle feature
NOISE_NO_FEATURE = ["labels", "duplicated"]

# ── Caricamento run già completati (resume) ───────────────────────────────
completed = set()
if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH, "r") as f:
        for line in f:
            try:
                r = json.loads(line)
                completed.add((r["noise_type"], r["feature"], r["percentage"], r["seed"]))
            except Exception:
                pass
    print(f"✅ Run già completati: {len(completed)}")
else:
    print("✅ Nessun risultato precedente — partenza da zero")

# ── Costruzione lista run da eseguire ─────────────────────────────────────
runs_to_do = []

for noise_type in NOISE_TYPES:
    if noise_type in NOISE_NO_FEATURE:
        # labels e duplicated non iterano sulle feature
        for percentage in NOISE_LEVELS:
            for seed in range(N_RUNS):
                key = (noise_type, "all_features", percentage, seed)
                if key not in completed:
                    runs_to_do.append((noise_type, "all_features", percentage, seed))
    else:
        for feature in NOISE_FEATURES:
            for percentage in NOISE_LEVELS:
                for seed in range(N_RUNS):
                    key = (noise_type, feature, percentage, seed)
                    if key not in completed:
                        runs_to_do.append((noise_type, feature, percentage, seed))

total_runs = (len(NOISE_NO_FEATURE) * len(NOISE_LEVELS) * N_RUNS) + \
             ((len(NOISE_TYPES) - len(NOISE_NO_FEATURE)) * len(NOISE_FEATURES) * len(NOISE_LEVELS) * N_RUNS)
remaining  = len(runs_to_do)

print(f"✅ Run totali    : {total_runs}")
print(f"✅ Run rimanenti : {remaining}")
print(f"✅ Stima tempo   : {remaining * 60 / 3600:.1f} ore")
print(f"✅ Avvio         : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# ── Loop principale ────────────────────────────────────────────────────────
with open(RESULTS_PATH, "a") as results_file:
    for i, (noise_type, feature, percentage, seed) in enumerate(runs_to_do):

        try:
            # Per labels e duplicated passa la prima NOISE_FEATURE — viene ignorata
            feature_arg = NOISE_FEATURES[0] if noise_type in NOISE_NO_FEATURE else feature

            result = run_experiment(
                pt         = pt_train,
                noise_type = noise_type,
                feature    = feature_arg,
                percentage = percentage,
                seed       = seed
            )

            # Sovrascrivi feature con il valore corretto per il salvataggio
            result["feature"]   = feature
            result["timestamp"] = datetime.now().isoformat()

            # Salvataggio incrementale
            results_file.write(json.dumps(result) + "\n")
            results_file.flush()

            # Log ogni 10 run
            if (i + 1) % 10 == 0:
                avg_duration = result["duration_s"]
                elapsed_h    = (i + 1) * avg_duration / 3600
                remaining_h  = (remaining - i - 1) * avg_duration / 3600
                print(f"[{datetime.now().strftime('%H:%M:%S')}] "
                    f"Run {i+1}/{remaining} | "
                    f"{noise_type:<12} | {feature:<25} | "
                    f"{percentage:.0%} | seed={seed} | "
                    f"F1={result['f1']:.4f} | "
                    f"elapsed≈{elapsed_h:.1f}h | "
                    f"remaining≈{remaining_h:.1f}h")

        except Exception as e:
            print(f"❌ ERRORE run ({noise_type}, {feature}, {percentage}, seed={seed}): {e}")
            continue

print(f"\n✅ Loop completato — {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"✅ Risultati salvati in: {RESULTS_PATH}")

[2026-03-03 08:26:20] [INFO] Esecuzione: missing (engine=Engine.SPARK)


✅ Run già completati: 860
✅ Run totali    : 1100
✅ Run rimanenti : 240
✅ Stima tempo   : 4.0 ore
✅ Avvio         : 2026-03-03 08:26:20



[Stage 26:=================================================>      (14 + 2) / 16]




[Stage 32:==============================>                          (9 + 8) / 17]




[Stage 47:=============>                                          (4 + 13) / 17]



...


[Stage 150828:=============================================>      (14 + 2) / 16]



[2026-03-03 10:09:16] [INFO] Esecuzione: missing (engine=Engine.SPARK)



[Stage 150869:============>                                       (4 + 13) / 17]




[Stage 151459:================================================>   (15 + 1) / 16]



[10:09:42] Run 240/240 | missing      | TP3_scaled                | 50% | seed=19 | F1=0.9746 | elapsed≈1.7h | remaining≈0.0h

✅ Loop completato — 2026-03-03 10:09:42
✅ Risultati salvati in: /home/PuckTrickadmin/Daniel/RESULTS/mlp_results.jsonl
